# NAMAA female reference — calm Egyptian narration
Uses `female_voice.wav` from the original NAMAA example (pinned commit) as the voice reference. Keeps the female conditioning across all segments, lower exaggeration 0.30, CFG 0.28, 0.48s pauses, plus mild tempo adjustment. Saves MP3/WAV and a QC report; publishes only a temporary synthetic test audio link. Does NOT touch Project 79 VPS.

In [ ]:
exec("import os,subprocess\nfrom pathlib import Path\npython=\"/kaggle/working/namaa_venv_v6/bin/python\"\nassert Path(python).exists(), \"Missing V6 runtime; do not reinstall automatically\"\nenv=os.environ.copy()\nenv.pop(\"PYTHONPATH\",None)\nenv.pop(\"PYTHONHOME\",None)\nenv[\"PYTHONNOUSERSITE\"]=\"1\"\nenv[\"HF_HOME\"]=\"/kaggle/working/namaa_hf_cache\"\nrun=Path(\"/kaggle/working/namaa_female_calm_v1_runner.py\")\nrun.write_text(\"import os,time,json,subprocess,hashlib\\nfrom pathlib import Path\\nimport torch,torchaudio\\nfrom huggingface_hub import hf_hub_download,snapshot_download\\nfrom safetensors.torch import load_file\\nfrom chatterbox.mtl_tts import ChatterboxMultilingualTTS\\n\\nroot=Path(\\\"/kaggle/working\\\")\\nout_dir=root/\\\"namaa_female_calm_v1\\\"\\nout_dir.mkdir(exist_ok=True)\\nsentences=[\\n\\\"تخيّلي إن فيه ملك، الناس لحد النهارده بتحكي عن ثروته. اسمه مانسا موسى.\\\",\\n\\\"مانسا موسى كان حاكم مالي، في غرب أفريقيا، من حوالي سبعمية سنة.\\\",\\n\\\"وقتها مالي كانت مركز مهم للتجارة، وخصوصًا تجارة الدهب والملح.\\\",\\n\\\"لكن اللي خلّى اسمه مشهور في أماكن بعيدة، كانت رحلة الحج بتاعته.\\\",\\n\\\"خرج في رحلة طويلة، ومعاه قافلة كبيرة، وناس كتير، وهدايا من الدهب.\\\",\\n\\\"ولما وصل القاهرة، فضل الناس يحكوا عن كرمه، وعن الدهب اللي وزّعه.\\\",\\n\\\"بس استني. مش كل رقم بنسمعه عن ثروته، نقدر نتأكد إنه صحيح.\\\",\\n\\\"لأننا بنتكلم عن زمن مختلف، مفيهوش بنوك، ولا حسابات زي دلوقتي.\\\",\\n\\\"والأكيد إنه ساب أثر كبير، في التجارة، وفي انتشار العلم والمعرفة.\\\",\\n\\\"وفي الآخر، السؤال المهم مش هو كان معاه كام.\\\",\\n\\\"السؤال هو، إزاي الثروة بتغيّر حياة الناس، وبتسيب أثر في التاريخ؟\\\",\\n\\\"ودي كانت حكاية مانسا موسى، بشكل بسيط، ومن غير مبالغات.\\\"\\n]\\nprint(\\\"GPU_READY\\\",torch.cuda.is_available(),torch.cuda.get_device_name(0),flush=True)\\nassert torch.cuda.is_available()\\nprint(\\\"DOWNLOAD_OFFICIAL_DEMO_FEMALE_REFERENCE\\\",flush=True)\\nreference=hf_hub_download(repo_id=\\\"omarelshehy/NAMAA-Egyptian-Voice\\\",repo_type=\\\"space\\\",revision=\\\"224ac11f27e97136523102e5f28d27dbcb30a34e\\\",filename=\\\"female_voice.wav\\\")\\nref_audio,ref_sr=torchaudio.load(reference)\\nprint(\\\"FEMALE_REFERENCE\\\",reference,\\\"REF_SEC\\\",round(ref_audio.shape[-1]/ref_sr,2),\\\"REF_BYTES\\\",Path(reference).stat().st_size,flush=True)\\nmodel=ChatterboxMultilingualTTS.from_pretrained(device=\\\"cuda\\\")\\nweights=snapshot_download(\\\"NAMAA-Space/NAMAA-Egyptian-TTS\\\",allow_patterns=[\\\"t3_mtl23ls_v2.safetensors\\\"])\\nmodel.t3.load_state_dict(load_file(str(Path(weights)/\\\"t3_mtl23ls_v2.safetensors\\\"),device=\\\"cpu\\\"),strict=True)\\nmodel.t3.to(\\\"cuda\\\").eval()\\nprint(\\\"NAMAA_WITH_FEMALE_REFERENCE_READY\\\",flush=True)\\ntorch.manual_seed(3701)\\naudio_parts=[]\\nlog=[]\\nstart=time.monotonic()\\ntotal_audio=0\\nfor index,text in enumerate(sentences,1):\\n    digest=hashlib.sha256((str(index)+\\\"|\\\"+text+\\\"|female_v1|cfg.28|ex.30\\\").encode()).hexdigest()[:10]\\n    clip=out_dir/f\\\"female_{index:02}_{digest}.wav\\\"\\n    now=time.monotonic()\\n    if clip.is_file() and clip.stat().st_size>2000:\\n        wav,sr=torchaudio.load(str(clip))\\n        assert sr==model.sr\\n        cached=True\\n    else:\\n        wav=model.generate(text,language_id=\\\"ar\\\",audio_prompt_path=reference if index==1 else None,exaggeration=0.30,cfg_weight=0.28,temperature=0.70,repetition_penalty=1.15)\\n        wav=wav.detach().cpu()\\n        torchaudio.save(str(clip),wav,model.sr)\\n        cached=False\\n    seconds=wav.shape[-1]/model.sr\\n    total_audio+=seconds\\n    audio_parts.append(wav)\\n    log.append({\\\"segment\\\":index,\\\"seconds\\\":round(seconds,2),\\\"text\\\":text,\\\"cached\\\":cached,\\\"elapsed\\\":round(time.monotonic()-now,2)})\\n    print(\\\"CLIP\\\",index,\\\"VOICE_FEMALE\\\",\\\"SECONDS\\\",round(seconds,2),\\\"CUMULATIVE\\\",round(total_audio,2),\\\"CACHED\\\",cached,flush=True)\\n    if total_audio+0.45*len(audio_parts)>=58:\\n        break\\npause=torch.zeros((1,int(model.sr*0.48)))\\ncombined=torch.cat([w for i,clip in enumerate(audio_parts) for w in ([clip] if i==0 else [pause,clip])],dim=-1)\\nraw=out_dir/\\\"female_calm_raw.wav\\\"\\ntorchaudio.save(str(raw),combined,model.sr)\\nmp3=out_dir/\\\"namaa_mansa_female_calm_v1.mp3\\\"\\nprocessed=out_dir/\\\"namaa_mansa_female_calm_v1.wav\\\"\\nsubprocess.run([\\\"ffmpeg\\\",\\\"-y\\\",\\\"-hide_banner\\\",\\\"-loglevel\\\",\\\"error\\\",\\\"-i\\\",str(raw),\\\"-af\\\",\\\"atempo=0.91,loudnorm=I=-17:TP=-2:LRA=10\\\",\\\"-ar\\\",\\\"24000\\\",\\\"-ac\\\",\\\"1\\\",str(processed)],check=True)\\nsubprocess.run([\\\"ffmpeg\\\",\\\"-y\\\",\\\"-hide_banner\\\",\\\"-loglevel\\\",\\\"error\\\",\\\"-i\\\",str(processed),\\\"-codec:a\\\",\\\"libmp3lame\\\",\\\"-qscale:a\\\",\\\"3\\\",str(mp3)],check=True)\\nprocessed_audio,processed_sr=torchaudio.load(str(processed))\\nd=round(processed_audio.shape[-1]/processed_sr,2)\\nassert d>35 and 100000<mp3.stat().st_size<20000000,(d,mp3.stat().st_size)\\nprint(\\\"FEMALE_CALM_SUCCESS\\\",\\\"SECONDS\\\",d,\\\"BYTES\\\",mp3.stat().st_size,\\\"MP3\\\",str(mp3),\\\"ELAPSED\\\",round(time.monotonic()-start,1),flush=True)\\nreport={\\\"mp3\\\":str(mp3),\\\"voice_reference\\\":reference,\\\"voice_reference_seconds\\\":round(ref_audio.shape[-1]/ref_sr,2),\\\"duration\\\":d,\\\"segments\\\":log,\\\"settings\\\":{\\\"exaggeration\\\":0.30,\\\"cfg_weight\\\":0.28,\\\"temperature\\\":0.70,\\\"atempo\\\":0.91,\\\"pause_seconds\\\":0.48}}\\n(out_dir/\\\"report.json\\\").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding=\\\"utf-8\\\")\\nimport requests\\nwith mp3.open(\\\"rb\\\") as f:\\n    response=requests.post(\\\"https://tmpfiles.org/api/v1/upload\\\",files={\\\"file\\\":(mp3.name,f,\\\"audio/mpeg\\\")},timeout=90)\\nprint(\\\"UPLOAD_HTTP\\\",response.status_code,flush=True)\\nprint(\\\"UPLOAD_BODY\\\",response.text[:450],flush=True)\\nresponse.raise_for_status()\\nu=response.json()[\\\"data\\\"][\\\"url\\\"]\\nprint(\\\"FEMALE_CALM_DOWNLOAD_URL\\\",u.replace(\\\"tmpfiles.org/\\\",\\\"tmpfiles.org/dl/\\\"),flush=True)\\n\",encoding=\"utf-8\")\nprint(\"RUN_FEMALE_REFERENCE_CLEAN\",flush=True)\nsubprocess.run([python,\"-u\",str(run)],check=True,env=env,timeout=840)\n")